%md
## Gold layer

Reads `permits.silver.permits`.

**Fact:** `fact_permits`, permits applied for from 2020 onward
**Dimensions:** `dim_date`, `dim_postal_area`, `dim_permit_type`, `dim_status`
**Summaries:** `approval_trend`, `dq_results`

A dimension is only created where it adds something the fact table does not have, which is why structure type and work type stay as plain columns.

Everything except `dq_results` is rebuilt from silver on each run, so rerunning gives the same result. `dq_results` is appended as a log.

In [0]:
from pyspark.sql import functions as F
import datetime

spark.sql("CREATE SCHEMA IF NOT EXISTS permits.gold")

target = "permits.gold.fact_permits"

In [0]:
silver = spark.table("permits.silver.permits")

gold = (silver
    .filter(F.col("APPLICATION_DATE") >= F.lit("2020-01-01").cast("date"))
    # Standalone trade permits have is_duplicate_of_main false, so they survive this.
    .filter(~F.col("is_duplicate_of_main"))
    .filter(~F.col("is_admin_record"))
)

print(f"gold rows: {gold.count():,}")

In [0]:
# Dropped: street parts (already in full_address), GEO_ID and WARD_GRID (internal
# City codes), CURRENT_USE and PROPOSED_USE (15,000+ spellings), BUILDER_NAME
# (5% filled), the raw cost text, the area columns, bronze tracking columns.
gold = gold.select(
    F.col("permit_key"),
    F.col("PERMIT_NUM").alias("permit_number"),
    F.col("REVISION_NUM").alias("revision_number"),
    F.col("project_num").alias("project_number"),
    F.col("PERMIT_TYPE").alias("permit_type"),
    F.col("STRUCTURE_TYPE").alias("structure_type"),
    F.col("WORK").alias("work_type"),
    F.col("STATUS").alias("status"),
    F.col("source_table").alias("source"),
    F.col("APPLICATION_DATE").alias("application_date"),
    F.col("ISSUED_DATE").alias("issued_date"),
    F.col("COMPLETED_DATE").alias("completed_date"),
    F.col("application_year"),
    F.col("days_to_issue"),
    F.col("days_to_complete"),
    F.col("est_const_cost_clean").alias("estimated_cost_cad"),
    F.col("DWELLING_UNITS_CREATED").alias("dwelling_units_created"),
    F.col("DWELLING_UNITS_LOST").alias("dwelling_units_lost"),
    F.col("POSTAL").alias("postal_area"),
    F.col("full_address").alias("address"),
    F.col("DESCRIPTION").alias("description"),
    F.col("is_secondary_type").alias("is_standalone_secondary"),
    F.col("is_cancelled"),
    F.col("is_issued"),
    F.col("has_bad_date"),
)

print(len(gold.columns), "columns")

In [0]:
# NOT null is null in Spark, and a filter drops nulls. Without this,
# WHERE NOT is_cancelled would silently lose rows with a missing status.
for flag in ["is_standalone_secondary", "is_cancelled", "is_issued", "has_bad_date"]:
    gold = gold.withColumn(flag, F.coalesce(F.col(flag), F.lit(False)))

# 38 rows have issue dates before their application date, which would give
# negative durations and drag down every average.
for d in ["days_to_issue", "days_to_complete"]:
    gold = gold.withColumn(d, F.when(F.col("has_bad_date"), None).otherwise(F.col(d)))

# Both "Application On Hold" and "Application on Hold" exist. Power BI matches
# text case insensitively and rejects the duplicate key when building relationships.
gold = gold.withColumn("status", F.initcap(F.col("status")))
gold = gold.withColumn("postal_area", F.upper(F.col("postal_area")))

In [0]:
# Overwrite rather than merge: gold is always silver filtered and renamed,
# so rebuilding it gives the same result every time.
(gold.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target))

print(f"wrote {spark.table(target).count():,} rows to {target}")

In [0]:
# The LLM layer reads these with the schema. Overwriting the table wipes them,
# which is why they live here rather than being set by hand in the UI.
column_docs = {
    "permit_key": "Unique row identifier: permit number, revision number and permit type joined with a pipe.",
    "permit_number": "City of Toronto permit number. Not unique on its own because permits get revised.",
    "revision_number": "Revision of the permit. 00 is the original application.",
    "project_number": "Permit number without its trailing suffix. Rows sharing it belong to the same project.",
    "permit_type": "Type of permit, for example New Houses or Small Residential Projects. Joins to dim_permit_type.",
    "structure_type": "Type of building, for example SFD Detached or Apartment Building.",
    "work_type": "Kind of work, for example New Building or Interior Alterations.",
    "status": "Current stage of the permit. Joins to dim_status, which groups statuses into broader stages.",
    "source": "Which City file the row came from: active (in progress) or cleared (finished, cancelled or abandoned).",
    "application_date": "Date the permit application was submitted. Joins to dim_date.",
    "issued_date": "Date the permit was issued. Null if refused, cancelled before issue, or still waiting.",
    "completed_date": "Date the work was completed or the permit was closed. Null if not finished.",
    "application_year": "Year of application_date.",
    "days_to_issue": "Days from application to issue. Null if not issued or dates are invalid. The two most recent years are incomplete because slow permits have not been decided yet, so averages for those years look artificially fast. Use approval_trend for year over year comparisons.",
    "days_to_complete": "Days from issue to completion. Null if not completed or dates are invalid.",
    "estimated_cost_cad": "Construction value in Canadian dollars as declared by the applicant, not appraised. Null for most plumbing, drain and HVAC permits.",
    "dwelling_units_created": "Number of homes the project creates.",
    "dwelling_units_lost": "Number of homes the project removes, for example through demolition.",
    "postal_area": "First three characters of the postal code, for example M5V. Joins to dim_postal_area for the region name.",
    "address": "Street address of the work. Does not contain the district or neighbourhood name.",
    "description": "Free text description of the work written by the applicant.",
    "is_standalone_secondary": "True for plumbing, drain, HVAC, partial or conditional permits that have no main building permit on the same project.",
    "is_cancelled": "True if the permit was cancelled or closed as dormant. Exclude these when measuring completed work.",
    "is_issued": "True if the permit has an issued date.",
    "has_bad_date": "True for rows with impossible dates, such as issued before applied. Their durations are already set to null.",
}

for col_name, text in column_docs.items():
    safe = text.replace("'", "\\'")
    spark.sql(f"ALTER TABLE {target} ALTER COLUMN {col_name} COMMENT '{safe}'")

spark.sql(f"""COMMENT ON TABLE {target} IS
  'City of Toronto building permits applied for from 2020 onward. One row per permit revision.
   Plumbing, drain and HVAC permits that duplicate a main permit on the same project are excluded.'""")

print("descriptions attached to", len(column_docs), "columns")

In [0]:
fact = spark.table(target)

def build_dim(col_name):
    dim = fact.select(col_name).where(F.col(col_name).isNotNull()).distinct()

    # Power BI treats text keys case insensitively, so two values differing only
    # by case become a duplicate and the relationship is rejected. Fail here instead.
    clashes = (dim.groupBy(F.lower(F.col(col_name)))
                  .count()
                  .filter(F.col("count") > 1)
                  .count())
    if clashes:
        raise ValueError(f"{col_name}: {clashes} values differ only by case")

    return dim

def save(df, name):
    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"permits.gold.{name}"))
    print(f"{name}: {spark.table(f'permits.gold.{name}').count():,} rows")

In [0]:
dim_permit_type = (build_dim("permit_type")
    .withColumn("is_secondary_type",
        F.col("permit_type").rlike("(?i)plumbing|mechanical|drain|partial permit|conditional permit")))
save(dim_permit_type, "dim_permit_type")

# Grouping statuses by name alone is guesswork, so the share of permits with an
# issue date decides it. New statuses then classify themselves.
# "Approved" looks pending but 98% have a completion date and none an issue date.
share_issued = fact.groupBy("status").agg(
    F.avg(F.col("is_issued").cast("int")).alias("share_issued"))

dim_status = (build_dim("status")
    .join(share_issued, "status", "left")
    .withColumn("stage",
        F.when(F.col("status").isin("Closed", "Approved"), "Completed")
         .when(F.col("status").isin("Cancelled", "Closed - Dormant", "Abandoned", "Superseded"), "Cancelled")
         .when(F.col("status").isin("Refused", "Refusal Notice"), "Refused")
         .when(F.col("share_issued") >= 0.5, "Issued, in progress")
         .otherwise("Before issue"))
    .drop("share_issued"))
save(dim_status, "dim_status")

display(spark.table("permits.gold.dim_status").orderBy("stage", "status"))

In [0]:
# Nothing in the source names a district, so "permits in Scarborough" depends
# on this table rather than on what the LLM happens to know about Toronto.
# M1, M2, M3 and M8 map cleanly to old boroughs. The rest cross those lines,
# so they are named broadly rather than pretending to be exact.
regions = spark.createDataFrame([
    ("M1", "Scarborough"),
    ("M2", "North York"),
    ("M3", "North York"),
    ("M4", "East and Central Toronto"),
    ("M5", "Downtown and Central Toronto"),
    ("M6", "West Toronto"),
    ("M7", "Central Toronto"),
    ("M8", "Etobicoke"),
    ("M9", "Etobicoke and Northwest"),
], ["prefix", "region"])

dim_postal_area = (build_dim("postal_area")
    .withColumn("prefix", F.substring("postal_area", 1, 2))
    # Left join so an unmapped area shows a null region instead of disappearing.
    .join(regions, "prefix", "left")
    .select("postal_area", "region"))
save(dim_postal_area, "dim_postal_area")

spark.sql("ALTER TABLE permits.gold.dim_postal_area ALTER COLUMN region COMMENT 'Approximate region from the first two characters of the postal code. Scarborough, North York and Etobicoke are reliable. Other regions are broad.'")

print(f"without a region: {spark.table('permits.gold.dim_postal_area').filter(F.col('region').isNull()).count()}")

In [0]:
# Range comes from the data so it never needs editing. The upper end runs to 2028
# because of the few bad completion dates, which keeps those rows joinable.
bounds = fact.select(
    F.least(F.min("application_date"), F.min("issued_date"), F.min("completed_date")).alias("lo"),
    F.greatest(F.max("application_date"), F.max("issued_date"), F.max("completed_date")).alias("hi"),
).first()

start = f"{bounds['lo'].year}-01-01"
end = f"{bounds['hi'].year}-12-31"

dim_date = (spark.sql(f"SELECT explode(sequence(to_date('{start}'), to_date('{end}'), interval 1 day)) AS date")
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("date").cast("string")))
    .withColumn("month_number", F.month("date"))          # for sorting month_name
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("month_short", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("day_of_week", F.dayofweek("date"))       # 1 is Sunday
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7)))
save(dim_date, "dim_date")

print(f"date range: {start} to {end}")

In [0]:
# A year is only comparable once its slow permits have been decided, which takes
# about a year beyond the year itself. Derived from today so it never goes stale.
last_complete_year = datetime.date.today().year - 2

trend = (spark.table(target)
    .filter(F.col("application_year").between(2020, last_complete_year))
    .filter(F.col("days_to_issue").isNotNull())
    .groupBy("application_year")
    .agg(
        F.count("*").alias("permits_issued"),
        # Median because a handful of multi year permits pull the average up by 40 days.
        F.percentile_approx("days_to_issue", 0.5).alias("median_days_to_issue"),
        F.round(F.avg("days_to_issue"), 1).alias("avg_days_to_issue"),
    )
    .orderBy("application_year"))
save(trend, "approval_trend")

display(trend)

In [0]:
g = spark.table(target)
total = g.count()

metrics = [
    ("gold_rows",          total),
    ("distinct_keys",      g.select("permit_key").distinct().count()),
    ("cost_coverage_pct",  round(g.filter(F.col("estimated_cost_cad").isNotNull()).count() / total * 100, 1)),
    ("null_postal",        g.filter(F.col("postal_area").isNull()).count()),
    ("postal_no_region",   spark.table("permits.gold.dim_postal_area").filter(F.col("region").isNull()).count()),
    ("not_issued",         g.filter(~F.col("is_issued")).count()),
    ("cancelled",          g.filter(F.col("is_cancelled")).count()),
    ("bad_dates",          g.filter(F.col("has_bad_date")).count()),
]

dq = (spark.createDataFrame([(m, float(v)) for m, v in metrics], ["metric", "value"])
      .withColumn("layer", F.lit("gold"))
      .withColumn("run_at", F.current_timestamp()))

# Appended, not overwritten. One number proves nothing, the same number over
# time is a monitor.
dq.write.mode("append").saveAsTable("permits.gold.dq_results")

display(dq)

In [0]:
tables = ["fact_permits", "dim_date", "dim_postal_area", "dim_permit_type",
          "dim_status", "approval_trend"]

for t in tables:
    print(f"{t:18} {spark.table(f'permits.gold.{t}').count():,}")

g = spark.table(target)
print(f"\ndistinct keys equal rows: {g.count() == g.select('permit_key').distinct().count()}")

In [0]:
%sql
CREATE OR REPLACE TABLE permits.gold.kpis AS

-- One row holding the years, so they are written once instead of five times
WITH years AS (
  SELECT 2025 AS last_year, 2024 AS trend_year
),

-- Everything that comes from the fact table, counted in one pass
totals AS (
  SELECT
    count_if(f.application_year = y.last_year) AS permits_last_year,
    count_if(f.source = 'active') AS in_progress,
    sum(CASE WHEN f.application_year = y.last_year AND f.is_cancelled = false
             THEN f.dwelling_units_created END) AS homes_last_year,
    sum(CASE WHEN f.application_year = y.last_year AND f.is_cancelled = false
             THEN f.estimated_cost_cad END) AS value_last_year
  FROM permits.gold.fact_permits f
  CROSS JOIN years y
),

-- The median comes from the trend table, which already caps the year range
median AS (
  SELECT t.median_days_to_issue
  FROM permits.gold.approval_trend t
  JOIN years y ON t.application_year = y.trend_year
)

-- One row per card
SELECT 1 AS ord, 'permits_last_year' AS kpi, 'Permits applied for in 2025' AS label,
       permits_last_year AS value, 'count' AS format
FROM totals
UNION ALL
SELECT 2, 'median_days_to_issue', 'Median days to issue (2024)',
       median_days_to_issue, 'days'
FROM median
UNION ALL
SELECT 3, 'homes_last_year', 'New homes permitted in 2025',
       homes_last_year, 'count'
FROM totals
UNION ALL
SELECT 4, 'in_progress', 'Permits currently in progress',
       in_progress, 'count'
FROM totals
UNION ALL
SELECT 5, 'value_last_year', 'Declared construction value 2025',
       value_last_year, 'currency'
FROM totals;

In [0]:
%sql
CREATE OR REPLACE TABLE permits.gold.ui_by_region AS

-- Attach the region name to every permit. Left join so permits with no
-- postal code become Unknown instead of disappearing.
WITH permits_with_region AS (
  SELECT
    coalesce(d.region, 'Unknown') AS region,
    f.dwelling_units_created,
    f.days_to_issue
  FROM permits.gold.fact_permits f
  LEFT JOIN permits.gold.dim_postal_area d
    ON f.postal_area = d.postal_area
)

SELECT
  region,
  count(*) AS permits,
  sum(dwelling_units_created) AS homes_created,
  percentile_approx(days_to_issue, 0.5) AS median_days_to_issue
FROM permits_with_region
GROUP BY region;

In [0]:
%sql
CREATE OR REPLACE TABLE permits.gold.ui_monthly AS

-- Turn each application date into a year and month label like 2025-03
WITH monthly AS (
  SELECT
    date_format(application_date, 'yyyy-MM') AS year_month,
    dwelling_units_created
  FROM permits.gold.fact_permits
)

SELECT
  year_month,
  count(*) AS permits,
  sum(dwelling_units_created) AS homes_created
FROM monthly
GROUP BY year_month;

In [0]:
%sql
SELECT * FROM permits.gold.kpis ORDER BY ord;

In [0]:
%sql
SELECT region, permits FROM permits.gold.ui_by_region ORDER BY permits DESC;